In [ ]:
# ==============================================================
# 00 – Config & Environment Setup
# Central configuration for the entire MARL Credit Decisioning project
# Run this notebook first before any other notebook
# ==============================================================

import os
import sys
import random
import numpy as np
import torch
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

print("=" * 60)
print("MARL Credit Decisioning – Configuration & Setup")
print("=" * 60)

# --------------------------------------------------------------
# 1. Reproducibility
# --------------------------------------------------------------
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✓ Random seed set to {seed}")

set_seed(SEED)

# --------------------------------------------------------------
# 2. Device Configuration
# --------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA Version: {torch.version.cuda}")

# --------------------------------------------------------------
# 3. Project Paths (Windows-friendly)
# --------------------------------------------------------------
# Automatically detect project root (assumes this notebook is inside /notebooks)
try:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
except:
    ROOT = Path(".")

DATA_RAW        = ROOT / "data" / "raw"
DATA_PROCESSED  = ROOT / "data" / "processed"
DATA_SYNTHETIC  = ROOT / "data" / "synthetic"
RESULTS         = ROOT / "results"
MODELS          = ROOT / "models"
NOTEBOOKS       = ROOT / "notebooks"

# Create directories if they don't exist
for p in [DATA_RAW, DATA_PROCESSED, DATA_SYNTHETIC, RESULTS, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Project Root     : {ROOT.resolve()}")
print(f"✓ Data Raw         : {DATA_RAW}")
print(f"✓ Data Processed   : {DATA_PROCESSED}")
print(f"✓ Data Synthetic   : {DATA_SYNTHETIC}")
print(f"✓ Results          : {RESULTS}")
print(f"✓ Models           : {MODELS}")

# --------------------------------------------------------------
# 4. Global Hyperparameters
# --------------------------------------------------------------
CONFIG = {
    # Reproducibility
    "seed": SEED,
    "device": str(DEVICE),

    # Data
    "test_size": 0.25,
    "n_samples_per_country": 7000,

    # CNN Encoder
    "cnn_emb_dim": 64,
    "cnn_epochs": 25,
    "cnn_batch_size": 128,
    "cnn_lr": 1e-3,

    # PPO / MARL
    "ppo_lr": 2.5e-4,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_eps": 0.2,
    "ppo_epochs": 4,
    "num_episodes_ppo": 60,
    "num_episodes_marl": 70,

    # Reward Design
    "cost_fn": 5.0,          # False Negative cost (approve bad customer)
    "cost_fp": 1.0,          # False Positive cost (reject good customer)
    "thin_file_bonus": 0.35,
    "thin_file_penalty": 0.25,

    # Business Assumptions
    "avg_loan": 5500,
    "interest_margin": 0.135,
    "lgd": 0.60,
    "operating_cost_per_loan": 120,
}

print("\n✓ Global CONFIG dictionary created")

# --------------------------------------------------------------
# 5. Utility Functions
# --------------------------------------------------------------
def get_config():
    """Return a copy of the global configuration"""
    return CONFIG.copy()

def print_config():
    print("\n=== Current Configuration ===")
    for k, v in CONFIG.items():
        print(f"  {k:30}: {v}")

def save_config():
    import json
    config_path = RESULTS / "project_config.json"
    with open(config_path, "w") as f:
        json.dump(CONFIG, f, indent=4)
    print(f"✓ Config saved to {config_path}")

# --------------------------------------------------------------
# 6. Quick Environment Check
# --------------------------------------------------------------
print("\n=== Environment Check ===")
print(f"Python        : {sys.version.split()[0]}")
print(f"NumPy         : {np.__version__}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Check if german.data exists
german_path = DATA_RAW / "german.data"
if german_path.exists():
    print(f"✓ german.data found at {german_path}")
else:
    print(f"⚠ german.data NOT found. Please place it at: {german_path}")

print("\n" + "=" * 60)
print("Configuration completed successfully.")
print("You can now run the notebooks in order: 01 → 10")
print("=" * 60)

# Optional: display full config
print_config()
save_config()